# Programming basics — companion notebook

**PSYC 81.09: Storytelling with Data**  
Jeremy R. Manning · Dartmouth College · Spring 2026

This notebook accompanies the [Programming basics lecture](programming-basics.html). We'll work through **two** realistic problems — the kinds of things you'll actually do in your Part II data stories. The goal is to *write* code, not just read it.

**How to use this notebook:**
- Run each cell with `Shift + Enter`
- Try modifying the examples — break them on purpose!
- Use AI tools (chat.dartmouth.edu, Claude Code) when you get stuck
- Read error messages carefully — they usually tell you exactly what's wrong

---

## Problem 1: exploring a real dataset

You have a list of songs, each a dict with `title`, `artist`, `genre`, and `plays`. **Your task:** find the three genres with the highest *average* plays per song.

This looks like a simple question, but it's a complete analysis pipeline: **group**, **aggregate**, **sort**, **slice**. You'll do some version of this in every Part II project.

In [ ]:
songs = [
    {'title': 'Blinding Lights',   'artist': 'The Weeknd',  'genre': 'pop',     'plays': 4_200_000_000},
    {'title': 'Shape of You',      'artist': 'Ed Sheeran',  'genre': 'pop',     'plays': 3_800_000_000},
    {'title': 'Someone Like You',  'artist': 'Adele',       'genre': 'pop',     'plays': 2_100_000_000},
    {'title': 'Bohemian Rhapsody', 'artist': 'Queen',       'genre': 'rock',    'plays': 2_400_000_000},
    {'title': 'Sweet Child OMine', 'artist': 'Guns N Roses','genre': 'rock',    'plays': 1_900_000_000},
    {'title': 'Hotel California',  'artist': 'Eagles',      'genre': 'rock',    'plays': 1_300_000_000},
    {'title': 'Lose Yourself',     'artist': 'Eminem',      'genre': 'hip-hop', 'plays': 1_600_000_000},
    {'title': 'God\'s Plan',       'artist': 'Drake',       'genre': 'hip-hop', 'plays': 2_500_000_000},
    {'title': 'HUMBLE.',           'artist': 'Kendrick',    'genre': 'hip-hop', 'plays': 1_800_000_000},
    {'title': 'Take Five',         'artist': 'Dave Brubeck','genre': 'jazz',    'plays':   450_000_000},
    {'title': 'So What',           'artist': 'Miles Davis', 'genre': 'jazz',    'plays':   380_000_000},
    {'title': 'Clair de Lune',     'artist': 'Debussy',     'genre': 'classical','plays':  620_000_000},
    {'title': 'Moonlight Sonata',  'artist': 'Beethoven',   'genre': 'classical','plays':  710_000_000},
]

print(f'{len(songs)} songs across {len(set(s["genre"] for s in songs))} genres')

### Step 1: group plays by genre

We need to collect all the play counts for each genre into a single place — a dictionary where the keys are genres and the values are lists of play counts.

In [ ]:
plays_by_genre = {}
for song in songs:
    g = song['genre']
    plays_by_genre.setdefault(g, []).append(song['plays'])

# Look at what we built
for genre, plays in plays_by_genre.items():
    print(f'{genre}: {len(plays)} songs, total plays = {sum(plays):,}')

**`setdefault` explained:** `d.setdefault(key, default)` returns the value at `key` if it exists, otherwise inserts `default` and returns it. It's a concise way to build up a dict-of-lists without writing `if key not in d: d[key] = []` every time.

### Step 2: compute the average per genre

Now a dictionary comprehension turns our dict-of-lists into a dict-of-averages.

In [ ]:
avg_by_genre = {
    g: sum(plays) / len(plays)
    for g, plays in plays_by_genre.items()
}

for genre, avg in avg_by_genre.items():
    print(f'{genre}: avg {avg:,.0f} plays per song')

### Step 3: sort by average plays, descending

The `key` argument of `sorted` lets us sort by any derived value. Using `-x[1]` (negate the value) gives us descending order.

In [ ]:
ranked = sorted(avg_by_genre.items(), key=lambda item: -item[1])
for genre, avg in ranked:
    print(f'{genre}: {avg:,.0f}')

### Step 4: take the top 3

In [ ]:
top_three = ranked[:3]
print('Top 3 genres by average plays:')
for i, (genre, avg) in enumerate(top_three, start=1):
    print(f'  {i}. {genre} ({avg:,.0f} plays/song)')

### Challenge: extend the analysis

Try one or more of the following. Each asks you to reuse the same pipeline with a twist:

1. Find the top 3 genres by **median** plays instead of mean. Does the ranking change? Why?
2. Find the top **artist** (not genre) by total plays across all their songs.
3. Print a bar-chart-style output using `█` characters, where each bar's length is proportional to the average plays for that genre.
4. Add more songs to the dataset. Does your pipeline still work without modification? (It should!)

In [ ]:
# Your turn:


---

## Problem 2: find and fix the sneaky bug

The function below is supposed to return a dict mapping each student to their average grade. It runs without errors — but the numbers are wrong. Can you find the bug?

In [ ]:
def student_averages(grades):
    result = {}
    for student, scores in grades.items():
        total = 0
        for s in scores:
            total += s
        result[student] = total / len(grades)
    return result

grades = {
    'Ada':   [90, 85, 92],
    'Grace': [78, 88],
    'Alan':  [95, 91, 88, 84],
}

print(student_averages(grades))

**Trace it by hand first.** What should Grace's average be? (78 + 88) / 2 = **83**. What does the function return?

**Debugging strategy:**

1. Verify by hand what the answer *should* be
2. Read the code one line at a time — what is each variable at each step?
3. Add `print` statements to show intermediate values
4. Ask AI: *"this function returns wrong numbers — walk through it step by step"*

### The fix

In [ ]:
def student_averages(grades):
    result = {}
    for student, scores in grades.items():
        total = 0
        for s in scores:
            total += s
        result[student] = total / len(scores)  # was len(grades) — wrong denominator
    return result

print(student_averages(grades))
# Expected: {'Ada': 89.0, 'Grace': 83.0, 'Alan': 89.5}

**Why this was sneaky:** Ada's average came out right *by coincidence* — she had 3 scores and there are 3 students, so `total / len(grades)` equaled `total / len(scores)`. The bug was invisible until you tested a student with a different number of scores.

**Lesson:** always test with inputs that have *different shapes*. If every input has the same size, coincidences can hide bugs.

### Challenge: write a more robust version

Extend `student_averages` to handle edge cases:

1. A student with an **empty list** of scores — what should happen?
2. A student whose scores include a non-number (e.g., `'absent'`)
3. An **empty `grades` dict** — what should happen?

There's more than one reasonable answer. What does the *caller* of this function need?

In [ ]:
# Your turn:


---

## Wrap-up

You've practiced the two patterns you'll reuse constantly:

1. **Group → aggregate → sort → slice** — the core of every data analysis
2. **Verify with small examples** — the fastest way to catch subtle bugs

Next up: **GitHub** — how to version, share, and collaborate on your code (and submit Assignment 3).